# PROJECT 2 
by Szymon Waliczek
 - Impact of spatial quantization and transverse confinement on 2DEG electron transport in **InAs**
 - With translational symmetry along **Y** axis(wraparound) and different make_system function.
 - **With spin**
 - If we set B=0, we only allow for Rashba effect
 - If we set B>>0 and alpha(Rashba coefficient = 0) we shall see Zeeman effect
 - If we set (B or g) and alpha  = 0 system should act as if the spin was degenerated(NO SPIN)

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import kwant
import numpy as np
from scipy.sparse.linalg import eigs
from matplotlib import pyplot as plt
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg')

# Pauli Matrices
import tinyarray
s_0 = tinyarray.array([[1, 0], [0, 1]])
s_x = tinyarray.array([[0, 1], [1, 0]])
s_y = tinyarray.array([[0, -1j], [1j, 0]])
s_z = tinyarray.array([[1, 0], [0, -1]])

In [ ]:
# Physical constants
from scipy.constants import physical_constants
eV = physical_constants['electron volt'][0]
h_bar = physical_constants['Planck constant over 2 pi'][0]
m_el = physical_constants['electron mass'][0]
mu_B = physical_constants['Bohr magneton in eV/T'][0]
mu = 0
g_factor = -14.7
a = 5e-9
m_eff = 0.023 * m_el
t = (h_bar**2 / (2 * m_eff * a**2)) / eV # hopping in eV
W, L = 100, 200
freedom_deg = 2; # Two spins
PI = np.pi

In [ ]:
print(f"h_bar = {h_bar}")
print(f"m_eff = {m_eff}")
print(f"mu_B = {mu_B}")
print(f"a = {a}")
print(f"t = {t}")
print(f"W, L = {W, L}")

In [ ]:
def rectangle_infinite(pos, width): # Shape infinite
    (x, y) = pos
    return abs(x) < width/2
#-----------------------------------------------------------------------
def rectangle(pos, width, length): # Shape finite
    (x, y) = pos
    return abs(x) < width/2 and abs(y) < length/2
#-----------------------------------------------------------------------
def onsite(site, mu, E_z):
    return (4*t - mu)*s_0 + E_z*s_z
#-----------------------------------------------------------------------
def hop(site1, site2, alpha):
    dx, dy = site1.pos - site2.pos
    if abs(dx) > a*0.5:
        return -t * s_0 + (1j * alpha / (2 * a*1e9)) * s_y
    if abs(dy) > a*0.5:
        return -t * s_0 - (1j * alpha / (2 * a*1e9)) * s_x
#-----------------------------------------------------------------------
def make_sys(a, width): # System infintie
    lat = kwant.lattice.square(a*1e9, norbs = freedom_deg)
    sym_y = kwant.TranslationalSymmetry((0, a*1e9))
    sys = kwant.Builder(sym_y) 
    sys[lat.shape(lambda p: rectangle_infinite(p, width), (0, 0))] = onsite
    sys[lat.neighbors(1)] = hop
    return sys
#-----------------------------------------------------------------------
def make_sysf(a, width, length): # System finite
    lat = kwant.lattice.square(a*1e9, norbs = freedom_deg)
    sys = kwant.Builder()
    sys[lat.shape(lambda p: rectangle(p, width, length), (0, 0))] = onsite
    sys[lat.neighbors(1)] = hop
    return sys, lat

In [ ]:
def plot_sys(sys): # Plot system
    kwant.plot(sys, fig_size=(2, 3.5), show=False)
    plt.title(f"Lattice InAs (a = {a*1e9}nm)") 
    plt.xlabel("x [nm]")
    plt.ylabel("y [nm]")
    plt.ticklabel_format(axis='both', useMathText=True)
    plt.show()

In [ ]:
sys = make_sys(a, W)
wrapped = kwant.wraparound.wraparound(sys, coordinate_names='y').finalized()
sysf, latf = make_sysf(a, W, L)
plot_sys(sys)
plot_sys(sysf)

In [ ]:
def lead_shape(pos, width):
    return abs(pos[0]) < width / 2

def make_lead(sys, lat, a, width):
    lead = kwant.Builder(kwant.TranslationalSymmetry((0, a*1e9)))
    lead[lat.shape(lambda p: lead_shape(p, width), (0, 0))] = onsite
    lead[lat.neighbors()] = hop
    sys.attach_lead(lead)           # Lead 0
    sys.attach_lead(lead.reversed()) # Lead 1

In [ ]:
make_lead(sysf, latf, a, W)
plot_sys(sysf)
fsysf = sysf.finalized()

In [ ]:
def eigen(wrapped_sys, mu, alpha, B, modes):
    E_z = 0.5*g_factor*mu_B*B
    k_range = np.linspace(-PI, PI, 200)
    E, V = [], []
    for i in k_range:
        params = dict(mu=mu, B=B, E_z=E_z, alpha=alpha, k_y=i)
        ham = wrapped_sys.hamiltonian_submatrix(params=params, sparse=True)
        eval, evec = eigs(ham, k=modes, sigma=0, which='LM')
        idx = eval.real.argsort() # Sortowanie skojarzone
        E.append(eval[idx].real)
        V.append(evec[:, idx])
    E = np.array(E)
    V = np.array(V)
    return E, k_range, V

In [ ]:
def plot_bands(E, k_range, modes):
    plt.figure(figsize=(4,3))
    for i in range(modes):
        plt.plot(k_range / (a*1e9), E[:, i], lw=1)  
    plt.title("Band structure")
    plt.ylabel("Energy [eV]")
    plt.xlabel("$k_y$ [$1/nm$]")
    plt.ylim(0,0.02)
    plt.xlim((-PI/3) / (a*1e9), (PI/3) / (a*1e9))
    plt.grid(True)
    plt.show()   

In [ ]:
Eval, k, Evec = eigen(wrapped, mu=0, alpha=0.1, B=0, modes=5)
plot_bands(E=Eval, k_range=k, modes=5)

In [ ]:
def plot_conductance(sys, Eb, Ef, n, mu, B, alpha):
    E_z = 0.5*g_factor*mu_B*B
    energies = np.linspace(Eb, Ef, n)
    transmissions = []
    params = dict(mu=mu, B=B, E_z=E_z, alpha=alpha)
    for E in energies:
        smatrix = kwant.smatrix(sys, E, params=params)
        transmissions.append(smatrix.transmission(1, 0))

    plt.figure(figsize=(4, 3))
    plt.plot(energies, transmissions, lw=1)
    plt.title(f"Conductance | B = {B} T") 
    plt.xlabel("Energy $E$ [eV]")
    plt.ylabel("Conductance $G$ [$2*e^2/h$]")
    plt.grid(True)
    plt.show()

In [ ]:
plot_conductance(fsysf, 0, 0.25, 100, mu=0.0, B=0.0, alpha=0)

In [ ]:
plot_conductance(fsysf, 0, 0.25, 100, mu=0.0, B=0.0, alpha=0)